In [1]:
import pandas as pd
from openai import OpenAI
import os
from dotenv import load_dotenv
from tqdm import tqdm
import time

In [2]:
# Load environment variables from the .env file
load_dotenv(".env")

True

In [3]:
df = pd.read_csv("googlemaps-scraper/data/newest_gm_reviews_2025-01-13.csv")

In [4]:
len(df)

31149

In [5]:
df.head(3)

,id_review,caption,relative_date,review_date,retrieval_date,rating,username,n_review_user,place_id
0,ChdDSUhNMG9nS0VJQ0FnSUNfMVBHRTh3RRAB,Saiyda,13 hours ago,2025-01-13 15:59:48.626486,2025-01-13 15:59:48.626537,5.0,kiruthika srinivasan,0,ChIJB0quoslnUjoRf_vm8BiGHsM
1,ChZDSUhNMG9nS0VJQ0FnSURmMzVfZElnEAE,Good waxing and face massage service by Shaila...,2 days ago,2025-01-11 15:59:48.626770,2025-01-13 15:59:48.626799,4.0,Yamini Srinivasan,0,ChIJB0quoslnUjoRf_vm8BiGHsM
2,ChZDSUhNMG9nS0VJQ0FnSUQtb05tZ2J3EAE,I've been coming here for the last 10 years an...,2 days ago,2025-01-11 15:59:48.626887,2025-01-13 15:59:48.626911,5.0,Pinky Kohli,9,ChIJB0quoslnUjoRf_vm8BiGHsM


In [6]:
df["retrieval_date"].value_counts()

retrieval_date
2025-01-13 15:59:48.626537    1
2025-01-13 19:38:52.743569    1
2025-01-13 19:39:05.674448    1
2025-01-13 19:39:05.674349    1
2025-01-13 19:39:05.674247    1
                             ..
2025-01-13 17:47:10.483319    1
2025-01-13 17:47:10.483182    1
2025-01-13 17:47:06.254778    1
2025-01-13 17:47:06.254673    1
2025-01-13 21:28:57.565981    1
Name: count, Length: 31149, dtype: int64

In [7]:
len(df["place_id"].value_counts())

43

In [8]:
df["review_date"].value_counts()

review_date
2025-01-13 15:59:48.626486    1
2024-06-13 19:38:52.743540    1
2024-05-13 19:39:05.674420    1
2024-05-13 19:39:05.674322    1
2024-05-13 19:39:05.674219    1
                             ..
2024-02-13 17:47:10.483285    1
2024-02-13 17:47:10.483099    1
2024-02-13 17:47:06.254749    1
2024-02-13 17:47:06.254644    1
2024-04-13 21:28:57.565938    1
Name: count, Length: 31149, dtype: int64

In [9]:
df['caption'].isna().sum()

np.int64(8093)

In [10]:
df = df[df['caption'].notna()]

In [11]:
len(df)

23056

In [12]:
df.head(3)

,id_review,caption,relative_date,review_date,retrieval_date,rating,username,n_review_user,place_id
0,ChdDSUhNMG9nS0VJQ0FnSUNfMVBHRTh3RRAB,Saiyda,13 hours ago,2025-01-13 15:59:48.626486,2025-01-13 15:59:48.626537,5.0,kiruthika srinivasan,0,ChIJB0quoslnUjoRf_vm8BiGHsM
1,ChZDSUhNMG9nS0VJQ0FnSURmMzVfZElnEAE,Good waxing and face massage service by Shaila...,2 days ago,2025-01-11 15:59:48.626770,2025-01-13 15:59:48.626799,4.0,Yamini Srinivasan,0,ChIJB0quoslnUjoRf_vm8BiGHsM
2,ChZDSUhNMG9nS0VJQ0FnSUQtb05tZ2J3EAE,I've been coming here for the last 10 years an...,2 days ago,2025-01-11 15:59:48.626887,2025-01-13 15:59:48.626911,5.0,Pinky Kohli,9,ChIJB0quoslnUjoRf_vm8BiGHsM


In [13]:
reviews = df["caption"]

In [14]:
len(reviews)

23056

In [15]:
reviews

0                                                   Saiyda
1        Good waxing and face massage service by Shaila...
2        I've been coming here for the last 10 years an...
5        Excellent team.. well done guys. 👍. Continue t...
7              Good and neat service by kalpana and rathna
                               ...                        
31144    Services was very good. Prabhakaran did a good...
31145    Jeevitha attended soo good and the service was...
31146                     Good service by geetha staff 😇 …
31147                Good service from jeevitha. Thank you
31148    Prabhakaran I did facial and beard trimmer ful...
Name: caption, Length: 23056, dtype: object

In [16]:
# Function to analyze sentiment
def analyze_sentiment(text):
    """
    Analyze the sentiment of a given text using OpenAI's GPT models.

    Args:
        text (str): The text to analyze.

    Returns:
        str: The sentiment analysis result (e.g., "Positive", "Negative", "Neutral").
    """
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
    
    try:
        response = client.chat.completions.create(
            model="gpt-4",  # Replace with the model you prefer
            messages=[
               {"role": "user", "content": f"""
                    Please classify the sentiment of the following text as Positive, Negative, Neutral, or Mixed, adhering to the following rules:
                    
                    1. If the text only contains a name (e.g., "John", "Sarah"), classify it as **Neutral**.
                    2. If there are spelling errors, correct them before performing the sentiment classification.
                    3. If the text is not in English, translate it to English before performing the sentiment classification.
                    4. Your classification output must strictly be one of the following: **Positive**, **Negative**, **Neutral**, or **Mixed**.
                    
                    Text: {text}
                    """}
            ],
            temperature=0.0
        )

        sentiment = response.choices[0].message.content
        return sentiment
        
    except Exception as e:
        print(f"Error during sentiment analysis: {e}")
        return None

In [17]:
response = analyze_sentiment(reviews[1])

In [18]:
response

'Positive'

In [19]:
reviews[1]

'Good waxing and face massage service by Shaila. Amazing hospitality'

In [20]:
analyze_sentiment("service was good, but bathroom was not clean")

'Mixed'

In [21]:
tqdm.pandas()

In [29]:
def analyze_sentiment_batch(texts):
    """
    Analyze sentiment for a batch of texts using OpenAI API.

    Args:
        texts (list of str): List of texts to analyze.

    Returns:
        list of str: List of sentiment analysis results.
    """
    from openai import OpenAI
    import os

    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
    
    try:
        results = []
        for text in texts:
            messages = [
                {
                    "role": "user",
                    "content": f"""
                    Please classify the sentiment of the following text as Positive, Negative, Neutral, or Mixed, adhering to the following rules:
                    
                    1. If the text only contains a name (e.g., "John", "Sarah"), classify it as **Neutral**.
                    2. If there are spelling errors, correct them before performing the sentiment classification.
                    3. If the text is not in English, translate it to English before performing the sentiment classification.
                    4. Your classification output must strictly be one of the following: **Positive**, **Negative**, **Neutral**, or **Mixed**.
                    
                    Text: {text}
                    """
                }
            ]
            
            response = client.chat.completions.create(
                model="gpt-4",  # Use "gpt-3.5-turbo" for faster response if accuracy is acceptable
                messages=messages,
                temperature=0.0
            )
            
            # Append the result to the results list
            result = response.choices[0].message.content.strip()
            results.append(result)
        
        return results

    except Exception as e:
        print(f"Error during sentiment analysis: {e}")
        return [None] * len(texts)

In [30]:
# tqdm.pandas()

# batch_size = 5

# sample_df['sentiment'] = None

# for i in tqdm(range(0, len(sample_df), batch_size), desc="Processing Batches"):
#     batch = sample_df['caption'].iloc[i:i+batch_size]
#     sentiments = analyze_sentiment_batch(batch.tolist())
#     sample_df.loc[batch.index, 'sentiment'] = sentiments

In [31]:
# List of texts
texts = df['caption'].tolist()  # Replace with your actual list if it's not from a DataFrame
batch_size = 5
sentiments = []

In [32]:
len(texts)

23056

In [33]:
from tqdm import tqdm

# Process the texts in batches
for i in tqdm(range(0, len(texts), batch_size), desc="Processing Batches"):
    batch = texts[i:i + batch_size]  # Extract batch of texts
    batch_sentiments = analyze_sentiment_batch(batch)  # Call your batch sentiment analysis function
    sentiments.extend(batch_sentiments)  # Append results to the sentiments list

Processing Batches:  66%|█████████▉     | 3043/4612 [2:37:04<1:37:58,  3.75s/it]

Error during sentiment analysis: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


Processing Batches:  66%|█████████▉     | 3045/4612 [2:37:13<1:44:28,  4.00s/it]

Error during sentiment analysis: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


Processing Batches:  66%|█████████▉     | 3047/4612 [2:37:25<2:10:47,  5.01s/it]

...

Error during sentiment analysis: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


Processing Batches: 100%|████████████████▉| 4611/4612 [3:20:32<00:01,  1.71s/it]

Error during sentiment analysis: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


Processing Batches: 100%|█████████████████| 4612/4612 [3:20:34<00:00,  2.61s/it]

Error during sentiment analysis: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


In [ ]:
# Add the sentiments back to the DataFrame if needed
sample_df['sentiment'] = sentiments

# Print the first few results for verification
print(sample_df.head())

In [29]:
sample_df.head()

,id_review,caption,relative_date,review_date,retrieval_date,rating,username,n_review_user,place_id,sentiment
1,ChZDSUhNMG9nS0VJQ0FnSUR2NVptekJnEAE,"Rosika did an excellent facial, great neck mas...",22 hours ago,2024-12-23 15:22:09.507031,2024-12-23 15:22:09.507073,5.0,Sujatha Madangopal,0,ChIJB0quoslnUjoRf_vm8BiGHsM,Positive
2,ChdDSUhNMG9nS0VJQ0FnSUR2NGJMZndBRRAB,had a great hair washing and blow dry experien...,2 days ago,2024-12-21 15:22:09.507740,2024-12-23 15:22:09.507785,5.0,080 harshitha sridhar,0,ChIJB0quoslnUjoRf_vm8BiGHsM,Positive
3,ChdDSUhNMG9nS0VJQ0FnSUR2dHN1WG9nRRAB,Excellent face massage by Goweri Thankyou you...,2 days ago,2024-12-21 15:22:09.507917,2024-12-23 15:22:09.507958,5.0,Varun Bala Vk,0,ChIJB0quoslnUjoRf_vm8BiGHsM,Positive
4,ChZDSUhNMG9nS0VJQ0FnSUNLdUtTR1NREAE,Aakash raj - Did very good haircut and shave s...,3 days ago,2024-12-20 15:22:09.508090,2024-12-23 15:22:09.508131,5.0,Rajesh Gopalsamy,23,ChIJB0quoslnUjoRf_vm8BiGHsM,Positive
5,ChZDSUhNMG9nS0VJQ0FnSUR2M01DbWR3EAE,Excellent hair spa service by Rosika,4 days ago,2024-12-19 15:22:09.508263,2024-12-23 15:22:09.508304,5.0,Kaviya,19,ChIJB0quoslnUjoRf_vm8BiGHsM,Positive


In [30]:
sample_df.to_csv("data/naturals_sentiments.csv")

In [31]:
len(sample_df)

46522

### Read Data

In [32]:
sentiments_df = pd.read_csv("data/naturals_sentiments.csv")

In [33]:
sentiments_df.head()

,Unnamed: 0,id_review,caption,relative_date,review_date,retrieval_date,rating,username,n_review_user,place_id,sentiment
0,1,ChZDSUhNMG9nS0VJQ0FnSUR2NVptekJnEAE,"Rosika did an excellent facial, great neck mas...",22 hours ago,2024-12-23 15:22:09.507031,2024-12-23 15:22:09.507073,5.0,Sujatha Madangopal,0,ChIJB0quoslnUjoRf_vm8BiGHsM,Positive
1,2,ChdDSUhNMG9nS0VJQ0FnSUR2NGJMZndBRRAB,had a great hair washing and blow dry experien...,2 days ago,2024-12-21 15:22:09.507740,2024-12-23 15:22:09.507785,5.0,080 harshitha sridhar,0,ChIJB0quoslnUjoRf_vm8BiGHsM,Positive
2,3,ChdDSUhNMG9nS0VJQ0FnSUR2dHN1WG9nRRAB,Excellent face massage by Goweri Thankyou you...,2 days ago,2024-12-21 15:22:09.507917,2024-12-23 15:22:09.507958,5.0,Varun Bala Vk,0,ChIJB0quoslnUjoRf_vm8BiGHsM,Positive
3,4,ChZDSUhNMG9nS0VJQ0FnSUNLdUtTR1NREAE,Aakash raj - Did very good haircut and shave s...,3 days ago,2024-12-20 15:22:09.508090,2024-12-23 15:22:09.508131,5.0,Rajesh Gopalsamy,23,ChIJB0quoslnUjoRf_vm8BiGHsM,Positive
4,5,ChZDSUhNMG9nS0VJQ0FnSUR2M01DbWR3EAE,Excellent hair spa service by Rosika,4 days ago,2024-12-19 15:22:09.508263,2024-12-23 15:22:09.508304,5.0,Kaviya,19,ChIJB0quoslnUjoRf_vm8BiGHsM,Positive


In [34]:
sentiments_df["sentiment"].value_counts()

sentiment
Positive                                                                                                       43866
Neutral                                                                                                         1220
Negative                                                                                                        1010
Mixed                                                                                                            294
Neutral (The sentiment is unclear due to missing information)                                                      2
                                                                                                               ...  
Neutral (The text "Pramila" is a name and does not express any sentiment)                                          1
Positive (The text "Bk hairstyle super service" expresses a positive sentiment towards a hairstyle service)        1
Neutral (The text "Gowsi" is a name and does not expre

In [35]:
# Get value counts of the 'sentiment' column
value_counts = sentiments_df["sentiment"].value_counts()

sentiments_df[sentiments_df["sentiment"].isin(value_counts[value_counts <= 2].index)]


,Unnamed: 0,id_review,caption,relative_date,review_date,retrieval_date,rating,username,n_review_user,place_id,sentiment
2375,2942,ChZDSUhNMG9nS0VJQ0FnSUR6aHN2WWV3EAE,I am statifed the hair services,6 months ago,2024-06-23 16:33:29.745855,2024-12-23 16:33:29.745903,5.0,Jee Vithya,0,ChIJ38JlxW1lUjoRVB-uo0b6CZQ,"Positive (Assuming ""statifed"" is a typo for ""s..."
4725,5572,ChdDSUhNMG9nS0VJQ0FnSURQbExDT21nRRAB,Hidrafacial nalla result kedachathu Beulah,3 weeks ago,2024-12-02 18:08:07.655470,2024-12-23 18:08:07.655511,4.0,Roja papitha,0,ChIJX2cOPjxnUjoRehfttXmstSA,Beulah's HydraFacial gave good results. - Posi...
4726,5574,ChZDSUhNMG9nS0VJQ0FnSUNQc2ZiakN3EAE,It's very good work Beulah ♥️,3 weeks ago,2024-12-02 18:08:07.655850,2024-12-23 18:08:07.655893,5.0,Arthipriya R,1,ChIJX2cOPjxnUjoRehfttXmstSA,It's very good work Beulah ♥️ - Positive
4727,5575,ChdDSUhNMG9nS0VJQ0FnSURua192V3VnRRAB,Had a hair cut which was fabulous ....great wo...,2 months ago,2024-10-23 18:08:07.656043,2024-12-23 18:08:07.656129,5.0,Bhoomi Mishra,0,ChIJX2cOPjxnUjoRehfttXmstSA,Had a hair cut which was fabulous ....great wo...
4728,5577,ChdDSUhNMG9nS0VJQ0FnSUNuMF9UanNBRRAB,I had a wonderful Hydra facial at *Naturals Sa...,2 months ago,2024-10-23 18:08:11.797140,2024-12-23 18:08:11.797206,5.0,Anitha Anitha,0,ChIJX2cOPjxnUjoRehfttXmstSA,I had a wonderful Hydra facial at *Naturals Sa...
...,...,...,...,...,...,...,...,...,...,...,...
45065,51539,ChdDSUhNMG9nS0VJQ0FnSUNYMDhMQzJRRRAB,I done facial..So much Satisfied.. Thank you m...,2 months ago,2024-10-24 13:41:58.731547,2024-12-24 13:41:58.731608,5.0,Bhuvanesh Kumar,0,ChIJu08uvNxvUjoRnU8vgmdp4qc,I had a facial. So much satisfaction. Thank yo...
45066,51540,ChZDSUhNMG9nS0VJQ0FnSUNYclktUGV3EAE,Latha this person did really well,2 months ago,2024-10-24 13:41:58.731767,2024-12-24 13:41:58.731820,5.0,Seneka Ravi,0,ChIJu08uvNxvUjoRnU8vgmdp4qc,"Latha, this person did really well. - Positive"
45067,51541,ChdDSUhNMG9nS0VJQ0FnSUNYclpYdm5BRRAB,Latha and Raji Serrvice was very good,2 months ago,2024-10-24 13:41:58.731974,2024-12-24 13:41:58.732026,5.0,Meena RM,0,ChIJu08uvNxvUjoRnU8vgmdp4qc,Latha and Raji's service was very good. - Posi...
45068,51542,ChZDSUhNMG9nS0VJQ0FnSUN4cjhfdGZBEAE,Wonderful service from Rajesh. Please maintain...,2 months ago,2024-10-24 13:41:58.732160,2024-12-24 13:41:58.732210,5.0,Manoj Karan,4,ChIJu08uvNxvUjoRnU8vgmdp4qc,Wonderful service from Rajesh. Please maintain...
